In [1]:
import jax
import jax.numpy as jnp
from typing import Callable, Sequence, Any
from functools import partial

jax.config.update('jax_enable_x64', True)

In [64]:
import jax
import jax.numpy as jnp
from typing import Callable, Sequence, Any
from functools import partial

def _get_zero_placeholder(func: Callable[..., jnp.ndarray]) -> Callable[..., jnp.ndarray]:
    """Returns a new function that returns zeros of the same shape as func."""
    def zero_placeholder_func(*args, **kwargs):
        return func(*args, **kwargs) * 0.0

    return zero_placeholder_func

def fill_intensity_matrix(
        intensity_matrix: Sequence[Sequence[Callable[..., jnp.ndarray]]],
        fill_func = Callable[..., jnp.ndarray]
) -> Sequence[Sequence[Callable[..., jnp.ndarray]]]:
    filled_matrix = [
        [
            (func if func is not None else fill_func)
            for func in row 
        ]
        for row in intensity_matrix
    ]
    
    return filled_matrix

def construct_func_from_intensity_matrix(
        intensity_matrix: Sequence[Sequence[Callable[..., jnp.ndarray]]]
) -> Callable[..., tuple[jnp.ndarray, jnp.ndarray]]:
    """
    Returns a new function that computes the outflow and inflow according to an
    n x n intensity matrix.
    """
    n = len(intensity_matrix)
    if not all(len(row) == n for row in intensity_matrix):
        raise ValueError("Intensity matrix must be square (n x n).")
    
    all_functions = (f for row in intensity_matrix for f in row)
    global_reference_func = next((f for f in all_functions if f is not None), None)
    
    if global_reference_func is None:
        raise ValueError("The intensity matrix contains only None entries. Cannot construct any function or determine output shape.")
    
    zero_func = _get_zero_placeholder(global_reference_func)
    intensity_matrix = fill_intensity_matrix(intensity_matrix, zero_func)
    
    @jax.jit
    def outflow_inflow(*args: Any, **kwargs: Any) -> tuple[jnp.ndarray, jnp.ndarray]:
        evaluated_matrix = [
            [func(*args, **kwargs)for func in row]
            for row in intensity_matrix
        ]

        outflow = [sum(row) for row in evaluated_matrix]
        outflow = jnp.stack(outflow, axis=-2)

        tranposed_matrix = list(zip(*evaluated_matrix))
        inflow = [jnp.stack(col, axis=-2) for col in tranposed_matrix]
        inflow = jnp.stack(inflow, axis=-3)       

        return outflow, inflow
    
    return outflow_inflow

@jax.jit
def next_p_j(p_j, outflow, inflow, step_size):
    dp_j = jnp.diff(p_j, axis=-1, prepend=0)
    outflow_integral = jnp.cumsum(outflow * dp_j, axis=-1)
    inflow_integral = jnp.sum(inflow * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
    inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)
    
    p_j = p_j + jnp.expand_dims(step_size, axis=-1) * (inflow_integral - outflow_integral)
    p_j = jnp.roll(p_j, shift=1, axis=-1)
    p_j = p_j.at[..., 0].set(0)
    
    return p_j

@partial(jax.jit, static_argnames=['flow'])
def solve_p_j(p_j_0: jnp.ndarray, 
              step_sizes: jnp.ndarray, 
              flow: Callable[..., tuple[jnp.ndarray, jnp.ndarray]], 
              *args: jnp.ndarray,
              **kwargs: jnp.ndarray):

    def scan_p_j(carry, step_size):
        p_j, t = carry
        outflow, inflow = flow(t, *args, **kwargs)
        p_j = next_p_j(p_j, outflow, inflow, step_size)
        t += step_size

        return (p_j, t), p_j
    
    t0 = jnp.zeros_like(step_sizes[0])
    _, p_j = jax.lax.scan(scan_p_j, (p_j_0, t0), step_sizes)
    
    p_j = jnp.concatenate([jnp.expand_dims(p_j_0, axis=0), p_j])
    p_j = jnp.swapaxes(p_j, 0, 1)
    p_j = jnp.swapaxes(p_j, 1, 2)

    return p_j

@partial(jax.jit, static_argnames=['flow'])
def solve(step_sizes: jnp.ndarray, 
          flow: Callable[..., tuple[jnp.ndarray, jnp.ndarray]], 
          *args: jnp.ndarray, 
          **kwargs: jnp.ndarray):

    outflow, inflow = flow(0, *args, **kwargs)
    step_sizes = jnp.expand_dims(step_sizes, axis=-1)
    
    p_j_0 = jnp.zeros_like(outflow)
    p_j_0 = p_j_0.at[(..., 0, slice(None))].set(1.0)

    p_j = solve_p_j(p_j_0, step_sizes, flow, *args, **kwargs)
    
    return p_j

@jax.jit
def next_p_j_heun(p_j, outflow, inflow, step_size):
    dp_j = jnp.diff(p_j, axis=-1, prepend=0)
    outflow_integral = jnp.cumsum(outflow * dp_j, axis=-1)
    inflow_integral = jnp.sum(inflow * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
    inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)
    
    delta_p_j = inflow_integral - outflow_integral
    
    p_j = p_j + jnp.expand_dims(step_size, axis=-1) * delta_p_j
    p_j = jnp.roll(p_j, shift=1, axis=-1)
    p_j = p_j.at[..., 0].set(0)
    
    return p_j, delta_p_j

@partial(jax.jit, static_argnames=['flow'])
def solve_p_j_heun(p_j_0: jnp.ndarray, 
                   step_sizes: jnp.ndarray, 
                   flow: Callable[..., tuple[jnp.ndarray, jnp.ndarray]], 
                   *args: jnp.ndarray,
                   **kwargs: jnp.ndarray):

    def scan_p_j_heun(carry, step_size):
        p_j, t, outflow, inflow = carry
        
        p_j_1, delta_p_j_1 = next_p_j_heun(p_j, outflow, inflow, step_size)
        
        t += step_size
        
        outflow, inflow = flow(t, *args, **kwargs)
        
        _, delta_p_j_2 = next_p_j_heun(p_j_1, outflow, inflow, step_size)
        
        step_size = jnp.expand_dims(step_size, axis=-1)
        p_j = p_j + 0.5 * step_size * delta_p_j_1
        p_j = jnp.roll(p_j, shift=1, axis=-1) + 0.5 * step_size * delta_p_j_2
        p_j = p_j.at[..., 0].set(0)

        return (p_j, t, outflow, inflow), p_j
    
    t0 = jnp.zeros_like(step_sizes[0])
    outflow_0, inflow_0 = flow(t0, *args, **kwargs)

    _, p_j = jax.lax.scan(scan_p_j_heun, (p_j_0, t0, outflow_0, inflow_0), step_sizes)
    
    p_j = jnp.concatenate([jnp.expand_dims(p_j_0, axis=0), p_j])
    p_j = jnp.swapaxes(p_j, 0, 1)
    p_j = jnp.swapaxes(p_j, 1, 2)

    return p_j

@partial(jax.jit, static_argnames=['flow'])
def solve_heun(step_sizes: jnp.ndarray, 
          flow: Callable[..., tuple[jnp.ndarray, jnp.ndarray]], 
          *args: jnp.ndarray, 
          **kwargs: jnp.ndarray):

    outflow, inflow = flow(0, *args, **kwargs)
    step_sizes = jnp.expand_dims(step_sizes, axis=-1)
    
    p_j_0 = jnp.zeros_like(outflow)
    p_j_0 = p_j_0.at[(..., 0, slice(None))].set(1.0)

    p_j = solve_p_j_heun(p_j_0, step_sizes, flow, *args, **kwargs)
    #p_j, t_0, oo, ii = solve_p_j_heun(p_j_0, step_sizes, flow, *args, **kwargs)
    
    return p_j


In [10]:
@jax.jit
def mu_1(t, u):
    return 0 * u + 0.15

@jax.jit
def mu_2(t, u):
    return 1.2 * jnp.exp(-0.05 * t + u * 0) * 0.25

In [87]:
u = jnp.concatenate([
    jnp.arange(0, 1, 1 / 128),
    jnp.arange(1, 2, 1 / 24),
    jnp.arange(2, 5, 1 / 12),
    jnp.arange(5, 30 + 1, 1)
])

#u = jax.device_put(u, device=jax.devices()[0])

u2 = jnp.expand_dims(u, axis=0) + jnp.expand_dims(jnp.arange(0, 1, 0.1), axis=1)
step_sizes = jnp.swapaxes(jnp.diff(u2), 1, 0)

intensity_matrix = [
    [None, mu_1, mu_1, mu_1, None],
    [None, None, mu_1, mu_1, mu_1],
    [None, None, None, None, None],
    [None, None, None, None, None],
    [None, None, None, None, None]
]

flow = construct_func_from_intensity_matrix(intensity_matrix)

In [90]:
p = solve(step_sizes, flow, u2)
ph = solve_heun(step_sizes, flow, u2)

In [91]:
p[0, 0, :, -1]

Array([1.00000000e+00, 9.96484375e-01, 9.92981110e-01, 9.89490160e-01,
       9.86011484e-01, 9.82545037e-01, 9.79090778e-01, 9.75648662e-01,
       9.72218647e-01, 9.68800691e-01, 9.65394751e-01, 9.62000785e-01,
       9.58618751e-01, 9.55248607e-01, 9.51890311e-01, 9.48543821e-01,
       9.45209097e-01, 9.41886096e-01, 9.38574778e-01, 9.35275101e-01,
       9.31987024e-01, 9.28710508e-01, 9.25445510e-01, 9.22191990e-01,
       9.18949909e-01, 9.15719226e-01, 9.12499900e-01, 9.09291893e-01,
       9.06095164e-01, 9.02909673e-01, 8.99735381e-01, 8.96572249e-01,
       8.93420237e-01, 8.90279307e-01, 8.87149418e-01, 8.84030534e-01,
       8.80922614e-01, 8.77825620e-01, 8.74739515e-01, 8.71664258e-01,
       8.68599814e-01, 8.65546143e-01, 8.62503207e-01, 8.59470969e-01,
       8.56449391e-01, 8.53438437e-01, 8.50438067e-01, 8.47448246e-01,
       8.44468935e-01, 8.41500099e-01, 8.38541701e-01, 8.35593702e-01,
       8.32656068e-01, 8.29728762e-01, 8.26811747e-01, 8.23904987e-01,
      

In [92]:
jnp.exp(-mu_1(0, u2) * u2)[0]

Array([1.        , 0.99882881, 0.99765899, 0.99649055, 0.99532347,
       0.99415776, 0.99299341, 0.99183043, 0.99066881, 0.98950855,
       0.98834965, 0.9871921 , 0.98603592, 0.98488108, 0.9837276 ,
       0.98257547, 0.98142469, 0.98027525, 0.97912717, 0.97798042,
       0.97683502, 0.97569097, 0.97454825, 0.97340687, 0.97226683,
       0.97112812, 0.96999074, 0.9688547 , 0.96771999, 0.96658661,
       0.96545455, 0.96432382, 0.96319442, 0.96206634, 0.96093957,
       0.95981413, 0.95869001, 0.9575672 , 0.95644571, 0.95532553,
       0.95420667, 0.95308911, 0.95197286, 0.95085792, 0.94974429,
       0.94863196, 0.94752093, 0.94641121, 0.94530278, 0.94419565,
       0.94308982, 0.94198529, 0.94088204, 0.93978009, 0.93867943,
       0.93758006, 0.93648198, 0.93538518, 0.93428967, 0.93319544,
       0.93210249, 0.93101082, 0.92992044, 0.92883132, 0.92774349,
       0.92665692, 0.92557163, 0.92448761, 0.92340487, 0.92232338,
       0.92124317, 0.92016422, 0.91908653, 0.91801011, 0.91693

In [78]:
ph[0, 0, :, -1]

Array([1.00000000e+00, 9.90668945e-01, 9.81424959e-01, 9.72267229e-01,
       9.63194951e-01, 9.54207326e-01, 9.45303565e-01, 9.36482886e-01,
       9.27744513e-01, 9.19087678e-01, 9.10511621e-01, 9.02015587e-01,
       8.93598830e-01, 8.85260611e-01, 8.77000195e-01, 8.68816859e-01,
       8.60709881e-01, 8.52678550e-01, 8.44722160e-01, 8.36840011e-01,
       8.29031411e-01, 8.21295674e-01, 8.13632119e-01, 8.06040073e-01,
       7.98518869e-01, 7.91067846e-01, 7.83686349e-01, 7.76373728e-01,
       7.69129343e-01, 7.61952555e-01, 7.54842734e-01, 7.47799255e-01,
       7.40821499e-01, 7.33908853e-01, 7.27060710e-01, 7.20276466e-01,
       7.13555527e-01, 7.06897302e-01, 7.00301204e-01, 6.93766655e-01,
       6.87293081e-01, 6.80879912e-01, 6.74526584e-01, 6.68232539e-01,
       6.61997225e-01, 6.55820093e-01, 6.49700600e-01, 6.43638208e-01,
       6.37632384e-01, 6.25788861e-01, 6.14165322e-01, 6.02757681e-01,
       5.91561928e-01, 5.80574127e-01, 5.69790416e-01, 5.59207004e-01,
      

In [42]:
step_size = jnp.expand_dims(step_sizes, axis=-1)[0]
pp2, dp2 = next_p_j_heun(pp, oo, ii, step_size)

In [60]:
dp3 = jnp.roll(dp2[0, 0, :], shift=1, axis=-1)
dp3 = dp3.at[..., 0].set(0)

Array([ 0.  , -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0.45,
       -0.45, -0.45,

In [43]:
t = step_size

outflow, inflow = flow(t, u2)

_, delta_p_j_2 = next_p_j_heun(pp2, outflow, inflow, step_size)

step_size = jnp.expand_dims(step_size, axis=-1)
pp2 = pp2 + 0.5 * step_size * dp2
pp2 = jnp.roll(pp2, shift=1, axis=-1) + 0.5 * step_size * delta_p_j_2
pp2 = pp2.at[..., 0].set(0)

In [51]:
pp2[0].shape

(5, 134)

In [8]:
%timeit -r 10 -n 10 solve(step_sizes, flow, u2).block_until_ready()

310 ms ± 11.5 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [7]:
%timeit -r 1 -n 10 solve(step_sizes, flow, u2).block_until_ready()

344 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


In [9]:
%timeit -r 2 -n 100 solve(step_sizes, flow, u2).block_until_ready()

IndexError: index is out of bounds for axis 0 with size 0

In [ ]:
next_p_j(p_j, oo, ii, uu).shape

In [ ]:
jnp.cumsum(oo * dp_j, axis=-1)

In [ ]:
inflow_integral = jnp.sum(ii * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)

In [ ]:
inflow_integral

In [ ]:
dp_j = jnp.diff(p_j, axis=-1, prepend=0)
outflow_integral = jnp.cumsum(oo * dp_j, axis=-1)
inflow_integral = jnp.sum(ii * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)

(p_j + uu * (inflow_integral - outflow_integral))[0]

In [ ]:
def test_func(carry, step_size):
    print(step_size.shape)
    return carry, step_size

In [ ]:
final, hist = jax.lax.scan(test_func, 0, jnp.expand_dims(u3, axis=-1))